# 04_langgraph_and_state_machines: Graph Orchestration & Checkpoints

This notebook builds a stateful agent graph workflow using LangGraph, demonstrating centralized state modifications, conditional transitions, and database checkpoints.


In [1]:
import os
from dotenv import load_dotenv
from typing import TypedDict, Annotated, Sequence
from langchain_core.messages import BaseMessage, HumanMessage, AIMessage
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langgraph.checkpoint.memory import MemorySaver

load_dotenv(dotenv_path=r"d:\Study\Prep\.env")

# 1. State definition
class AgentState(TypedDict):
    messages: Annotated[list, add_messages]

# 2. Node definitions
def call_model(state: AgentState):
    user_msg = state["messages"][-1].content
    # Simple mock model logic
    if "approve" in user_msg.lower():
        reply = AIMessage(content="Approved transaction.")
    else:
        reply = AIMessage(content="Require human approval. Respond with 'approve' to execute.")
    return {"messages": [reply]}

# 3. Create graph
builder = StateGraph(AgentState)
builder.add_node("agent", call_model)
builder.add_edge(START, "agent")
builder.add_edge("agent", END)

# Persistent checkpointer
memory = MemorySaver()
graph = builder.compile(checkpointer=memory)

# 4. Execute with Thread ID
config = {"configurable": {"thread_id": "thread_1"}}
event1 = graph.invoke({"messages": [HumanMessage(content="Start transaction")]}, config)
print("Turn 1 Response:", event1["messages"][-1].content)

# Turn 2: Resuming context from thread checkpoint
event2 = graph.invoke({"messages": [HumanMessage(content="Yes, approve transaction")]}, config)
print("\nTurn 2 Response:", event2["messages"][-1].content)


Turn 1 Response: Require human approval. Respond with 'approve' to execute.

Turn 2 Response: Approved transaction.


### Output Explanation
- The `StateGraph` defines states and transition edges.
- The `MemorySaver` saves checkpoints across thread scopes, allowing subsequent inputs matching `thread_id: "thread_1"` to inherit historical conversation steps automatically.
